# ACS3953 Assignment 4

In this assignment, you will explore three game search algorithms introduced in class to solve the TicTacToe game. Enjoy working on Assignment 4!

Your name:

Your Student ID:

## TicTacToe Class (game environment)

In [70]:
class TicTacToe:
    def __init__(self):
        # Board is a list of 9 positions: '.', 'X', or 'O'
        self.initial = ['.'] * 9
        self.size = 3

    def to_move(self, state):
        # X moves first, alternate turns
        x_count = state.count('X')
        o_count = state.count('O')
        return 'X' if x_count == o_count else 'O'

    def actions(self, state):
        # All empty cells are legal actions
        return [i for i, cell in enumerate(state) if cell == '.']

    def result(self, state, action):
        # Return a new state after making a move
        new_state = state.copy()
        new_state[action] = self.to_move(state)
        return new_state

    def is_terminal(self, state):
        # The game ends if there's a winner or no empty cells
        return self.winner(state) is not None or '.' not in state

    def utility(self, state, player):
        # +1 if 'player' wins, -1 if loses, 0 otherwise
        winner = self.winner(state)
        if winner == player:
            return 1
        elif winner is not None:
            return -1
        else:
            return 0

    def winner(self, state):
        # Check all winning combinations
        lines = [
            (0,1,2), (3,4,5), (6,7,8),
            (0,3,6), (1,4,7), (2,5,8),
            (0,4,8), (2,4,6)
        ]
        for (i, j, k) in lines:
            if state[i] != '.' and state[i] == state[j] == state[k]:
                return state[i]
        return None

    def display(self, state):
        for i in range(0, 9, 3):
            print(' '.join(state[i:i+3]))
        print()



## <font color='red'>Q1: Minimax Search</font> 

Complete the code to make blow `minimax_search()`function implementable.

In [71]:
PREF_ORDER = [4, 0, 2, 6, 8, 1, 3, 5, 7]  # prefer center, corners, sides

def minimax_search(game, state):
    """Return the best action for the current player using Minimax."""
    player = game.to_move(state)

    def max_value(game, state):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = float('-inf'), None
        legal = set(game.actions(state))
        for a in [m for m in PREF_ORDER if m in legal]:
            v2, _ = min_value(game, game.result(state, a))
            if v2 > v:
                v, move = v2, a
        return v, move

    def min_value(game, state):
        # put your code here



        return v, move

    value, best_move = max_value(game, state)
    return best_move

## <font color='red'>Q2: α–β search </font> 

Complete the code to make blow `alpha_beta_search()`function implementable.

In [72]:
def alpha_beta_search(game, state):
    """
    Return best action for the current player using Alpha–Beta pruning.
    Matches textbook MAX-VALUE(game, state, α, β) / MIN-VALUE(game, state, α, β).
    """
    player = game.to_move(state)

    def ordered_actions(s):
        legal = set(game.actions(s))
        # reorder to improve pruning; fallback to natural order if needed
        ordered = [a for a in PREF_ORDER if a in legal]
        if len(ordered) < len(legal):
            # include any remaining legal moves not in PREF_ORDER
            ordered += [a for a in game.actions(s) if a not in ordered]
        return ordered

    def max_value(game, state, alpha, beta):
        # put your code here



        return v, move

    def min_value(game, state, alpha, beta):
        # put your code here




        return v, move

    _, best_move = max_value(game, state, float("-inf"), float("inf"))
    return best_move


## <font color='red'>Q3: MCTS</font> 

Complete the code for `uct_child()` and `_rollout()` to make `mcts_search()` function implementable.

In [73]:
import math

def mcts_search(game, root_state, time_limit_ms=300, C=math.sqrt(2), rollout_policy=None):
    """
    Monte-Carlo Tree Search (UCT variant).
    Returns: the action to play from root_state.

    Parameters
    ----------
    game : object exposing methods:
        - actions(state) -> list[action]
        - result(state, action) -> state'
        - is_terminal(state) -> bool
        - utility(state, player) -> {-1,0,1}
        - to_move(state) -> 'X' or 'O'
    root_state : current state
    time_limit_ms : per-decision budget (milliseconds)
    C : exploration constant (larger => more exploration)
    rollout_policy : None or callable(state)->action; if None, use random
    """

    # ---- small node structure kept inside the function ----
    class Node:
        __slots__ = ("state", "parent", "action_from_parent",
                     "children", "untried_actions", "visits", "value")
        def __init__(self, state, parent=None, action_from_parent=None):
            self.state = state
            self.parent = parent
            self.action_from_parent = action_from_parent
            self.children = []                         # list[Node]
            self.untried_actions = list(game.actions(state))  # lazy expansion
            self.visits = 0
            self.value = 0.0                          # total reward from root player's view

        def fully_expanded(self):
            return len(self.untried_actions) == 0

        def best_child_by_visits(self):
            return max(self.children, key=lambda c: c.visits)

        def uct_child(self, C):
            # Upper Confidence bound applied to Trees (UCB1):
            # score = Q/N + C * sqrt(ln(N_parent) / N_child)
            # put your code here



            return max(self.children, key=score)

    # ---- setup ----
    root_player = game.to_move(root_state)            # we always evaluate rewards for this player
    root = Node(root_state)
    deadline = time.perf_counter() + time_limit_ms / 1000.0

    # ---- MCTS main loop ----
    while time.perf_counter() < deadline:
        node = root

        # 1) SELECTION: descend with UCT while node is fully expanded & nonterminal
        while (not game.is_terminal(node.state)) and node.fully_expanded() and node.children:
            node = node.uct_child(C)

        # 2) EXPANSION: if nonterminal and there is an untried action, expand one
        if (not game.is_terminal(node.state)) and node.untried_actions:
            a = node.untried_actions.pop()            # choose one action to expand
            child_state = game.result(node.state, a)
            child = Node(child_state, parent=node, action_from_parent=a)
            node.children.append(child)
            node = child                              # continue from the new child

        # 3) SIMULATION (ROLLOUT): play randomly (or with a given policy) to the end
        reward = _rollout(game, node.state, root_player, rollout_policy)

        # 4) BACKPROPAGATION: update visits and value along the path to the root
        while node is not None:
            node.visits += 1
            node.value  += reward                     # reward is from root_player's perspective
            node = node.parent

    # choose the move whose child got the most visits (robust)
    if not root.children:                             # no legal actions (terminal)
        return None
    return root.best_child_by_visits().action_from_parent


def _rollout(game, state, root_player, policy):
    """
    Play from 'state' to a terminal state and return the utility for 'root_player'.
    If 'policy' is provided, it will be called as policy(state)->action; otherwise random.
    """
    # put your code here



    return game.utility(s, root_player)

## Test three searches on TicTacToe game

You can change the plies and seed to test the three searches.

In [76]:
game = TicTacToe()

# A random midgame state
def random_midgame_state(game, plies, seed):
    rng = random.Random(seed)
    s = game.initial[:]
    for _ in range(plies):
        if game.is_terminal(s): break
        acts = game.actions(s)
        s = game.result(s, rng.choice(acts))
    return s

mid = random_midgame_state(game, plies=3, seed=3)
print("Mid-game state:")
game.display(mid)

move1 = mcts_search(game, mid, time_limit_ms=300, C=1.414)
print("MCTS move from mid-game:", move1)
game.display(game.result(mid, move1) if move1 is not None else mid)

move2 = alpha_beta_search(game, mid)
print("alpha beta search move from mid-game:", move2)
game.display(game.result(mid, move2) if move2 is not None else mid)

move3 = minimax_search(game, mid)
print("alpha beta search move from mid-game:", move3)
game.display(game.result(mid, move3) if move3 is not None else mid)

Mid-game state:
. . O
X X .
. . .

MCTS move from mid-game: 0
O . O
X X .
. . .

alpha beta search move from mid-game: 5
. . O
X X O
. . .

alpha beta search move from mid-game: 5
. . O
X X O
. . .



## Campare three searches

In [77]:
import time, random

class InstrumentedGame:
    """
    Wraps your TicTacToe instance and counts node 'expansions' as the number
    of times actions(state) is called by the search — a good proxy for nodes expanded.
    """
    def __init__(self, base_game):
        self.g = base_game
        self.nodes = 0

    # delegate methods
    @property
    def initial(self): return self.g.initial
    def to_move(self, state): return self.g.to_move(state)
    def result(self, state, action): return self.g.result(state, action)
    def is_terminal(self, state): return self.g.is_terminal(state)
    def utility(self, state, player): return self.g.utility(state, player)
    def winner(self, state): return self.g.winner(state)
    def display(self, state): return self.g.display(state)

    # count expansions whenever actions() is requested
    def actions(self, state):
        self.nodes += 1
        return self.g.actions(state)

def random_midgame_state(game, moves=3, seed=0):
    """Generate a mid-game state by playing `moves` random legal moves."""
    rng = random.Random(seed)
    s = game.initial[:]
    for _ in range(moves):
        if game.is_terminal(s): break
        acts = game.actions(s)
        if not acts: break
        s = game.result(s, rng.choice(acts))
    return s

def run_with_stats(search_fn, base_game, state):
    """Run a search function on a state with timing and node counting."""
    G = InstrumentedGame(base_game)         # wrap to count expansions
    t0 = time.perf_counter()
    move = search_fn(G, state)
    ms = (time.perf_counter() - t0) * 1000.0
    return move, ms, G.nodes

# ---------- Comparison ----------
game = TicTacToe()

tests = [("Initial", game.initial)]
# add three random mid-game states of increasing length
tests += [(f"Mid-{k} plies", random_midgame_state(game, moves=k, seed=100+k))
          for k in (3, 4, 5)]

print(f"{'State':<12} | {'Alg':<11} | {'BestMove':<8} | {'Time':>8} | {'Nodes':>6}")
print("-"*60)
for name, st in tests:
    m_move, m_ms, m_nodes = run_with_stats(minimax_search, game, st)
    a_move, a_ms, a_nodes = run_with_stats(alpha_beta_search, game, st)
    t_move, t_ms, t_nodes = run_with_stats(mcts_search, game, st)

    print(f"{name:<12} | {'Minimax':<11} | {str(m_move):<8} | {m_ms:7.2f} | {m_nodes:6d}")
    print(f"{name:<12} | {'AlphaBeta':<11} | {str(a_move):<8} | {a_ms:7.2f} | {a_nodes:6d}")
    print(f"{name:<12} | {'mcts_search':<11} | {str(t_move):<8} | {t_ms:7.2f} | {t_nodes:6d}")

    if m_move != a_move:
        print(f"  ⚠️ Different choices on {name}: minimax={m_move}, alpha-beta={a_move}")
    print("-"*60)

State        | Alg         | BestMove |     Time |  Nodes
------------------------------------------------------------
Initial      | Minimax     | 4        | 1678.20 | 294778
Initial      | AlphaBeta   | 4        |   23.11 |   4382
Initial      | mcts_search | 4        |  300.10 |  15212
------------------------------------------------------------
Mid-3 plies  | Minimax     | 4        |    3.00 |    546
Mid-3 plies  | AlphaBeta   | 4        |    0.52 |     98
Mid-3 plies  | mcts_search | 4        |  300.03 |    377
------------------------------------------------------------
Mid-4 plies  | Minimax     | 6        |    0.54 |     90
Mid-4 plies  | AlphaBeta   | 6        |    0.08 |     14
Mid-4 plies  | mcts_search | 6        |  300.04 |    205
------------------------------------------------------------
Mid-5 plies  | Minimax     | 2        |    0.11 |     17
Mid-5 plies  | AlphaBeta   | 2        |    0.05 |      8
Mid-5 plies  | mcts_search | 7        |  300.06 |     49
--------------